In [3]:
import copy

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils import (create_transition_open, get_complete_DR, save_model_weights,
                   sample_transition, make_maze)
from networks.repair import RepairNetworkLinear


ROWS, COLS = 20, 20
NUM_STATES = ROWS * COLS
COST = 0.1
# Hyperparameters
RANK = 20
LR = 1e-3
TAU = 0.005
NUM_STEPS = 25000
BATCH_SIZE = 256
GAMMA = np.exp(-COST)  # discount = exp(-c) since cost is uniform

In [4]:
T_os = create_transition_open(shape=(ROWS, COLS))
D_os = get_complete_DR(T=T_os)

### Test TD definition

In [5]:
s = 50  # pick any state
neighbors = np.where(T_os[s] > 0)[0]
expected_D = sum(T_os[s, sp] * D_os[sp, :] for sp in neighbors)
lhs = D_os[s, :]
rhs = GAMMA * np.eye(NUM_STATES)[s] + GAMMA * expected_D
print(f"Max difference: {np.abs(lhs - rhs).max()}")  # should be ~1e-15

Max difference: 2.220446049250313e-16


### Test eigenbasis definition

In [6]:
T_os = create_transition_open(shape=(ROWS, COLS))
D_os = get_complete_DR(T=T_os)

T_new, barrier_states = make_maze(T_os, 'horizontal', ROWS, COLS)
D_true = get_complete_DR(T=T_new)

## Check how well top-k eigenvectors of D_os can represent D_true
eigenvalues, V_full = np.linalg.eigh(D_os)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
V_full = V_full[:, idx]

print(f"\n{'k':>5} | {'Projection Error':>18} | {'Baseline Error (D_os)':>22}")
print("-" * 55)

baseline_error = np.linalg.norm(D_os - D_true) / np.linalg.norm(D_true)

for k in [5, 10, 20, 50, 100, 200, 400]:
    V_k = V_full[:, :k]
    # Best possible P in the k-dimensional subspace
    P_best = V_k.T @ D_true @ V_k
    D_reconstructed = V_k @ P_best @ V_k.T
    error = np.linalg.norm(D_reconstructed - D_true) / np.linalg.norm(D_true)
    print(f"{k:>5} | {error:>18.6f} | {baseline_error:>22.6f}")


    k |   Projection Error |  Baseline Error (D_os)
-------------------------------------------------------
    5 |           0.873421 |               0.157820
   10 |           0.787377 |               0.157820
   20 |           0.667087 |               0.157820
   50 |           0.503480 |               0.157820
  100 |           0.385747 |               0.157820
  200 |           0.258325 |               0.157820
  400 |           0.000000 |               0.157820


In [7]:
# How well does k=50 represent D_os itself?
eigenvalues_k = eigenvalues[idx]
D_os_approx = V_k @ np.diag(eigenvalues_k) @ V_k.T
dos_self_error = np.linalg.norm(D_os_approx - D_os) / np.linalg.norm(D_os)
print(f"D_os self-reconstruction error at k=50: {dos_self_error:.6f}")

D_os self-reconstruction error at k=50: 1.165310
